# Obligatorio TIA 2026 - Parte 3 (opcional)

- Estudiante 1: `nombre apellido - nro estudiante`
- Estudiante 2: `nombre apellido - nro estudiante`
- Estudiante 3: `nombre apellido - nro estudiante`

**Objetivo:** reward medio >= 40 en `highway-fast-v0` en el menor wall-time posible.

**Mejoras aplicadas:** _TODO_ (hiperparams, vec env, n-step, dueling, etc.)

In [ ]:
import time, json, torch, numpy as np, random
import highway_env
from utils import make_env


In [ ]:
# Reproducibilidad
# Pin de seeds en todos los RNGs que tocamos. El env tambien debe sembrarse:
# make_env(seed=SEED) abajo, y para envs vectorizados sembrar cada worker con
# SEED + worker_idx para que no compartan trayectorias.
SEED = 23
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
ENV_NAME = "highway-fast-v0"
TARGET_REWARD = 40.0
EPISODE_BLOCK = 50

# Cap del wall-time para que la re-ejecucion del profesor no se cuelgue si la
# corrida no converge. El loop de entrenamiento debe consultar este budget y
# cortar limpio (ej. retornar las rewards acumuladas hasta el momento).
MAX_WALL_TIME_MIN = 30 # correr como maximo 30 minutos, sino cortar y reportar lo que se tenga
N_EVAL_EPISODES = 100  # episodios greedy para la evaluacion final

# NOTA: NO modificar estos valores.

In [ ]:
# La consigna autoriza cambiar el preprocesado de imagen y el stacking en
# Parte 3. Restriccion: train_env y eval_env DEBEN usar los mismos valores —
# si entrenan con un preprocesado y evaluan con otro, el reward no compara nada.
OBS_HEIGHT = 84
OBS_WIDTH = 336
NUM_STACKED_FRAMES = 4

# NOTA: SI modificar estos valores.
# Ademas de estos hiperparametros, pueden modificar el ambiente en cuanto a su
# observacion (ej. preprocesado, stacking), pero NUNCA modificar la recompensa ni 
# la dinamica del ambiente (cantidad de carriles, cantidad de autos, etc).

In [ ]:
# Estos definen la DIFICULTAD del benchmark y la dinamica del simulador, NO
# son palancas del agente. Bajar VEHICLES_COUNT o subir LANES_COUNT hace el
# problema mas facil (menos congestion = menos colisiones); cambiar
# SIM_FREQUENCY_TRAIN o EPISODE_DURATION cambia la dinamica del simulador y
# el largo del problema. La competencia mide ALGORITMO. Cualquier
# entrega que los modifique queda fuera del ranking.
LANES_COUNT = 3
VEHICLES_COUNT = 30
VEHICLES_DENSITY = 1.0
HIGH_SPEED_REWARD = 0.8
SIM_FREQUENCY_TRAIN = 5
SIM_FREQUENCY_EVAL = 30
EPISODE_DURATION = 120

# NOTA: NO modificar estos valores.

In [ ]:
def process_state(obs):
    t = torch.from_numpy(np.asarray(obs, dtype=np.uint8)).float().div_(255.0)
    return t.unsqueeze(0) if t.ndim == 3 else t

# from my_super_agent_model import MY_SUPER_AGENT_Model

train_env = make_env(
    ENV_NAME,
    obs_height=OBS_HEIGHT,
    obs_width=OBS_WIDTH,
    stack_frames=NUM_STACKED_FRAMES,
    simulation_frequency=SIM_FREQUENCY_TRAIN,
    seed=SEED,
    duration=EPISODE_DURATION,
    lanes_count=LANES_COUNT,
    vehicles_count=VEHICLES_COUNT,
    vehicles_density=VEHICLES_DENSITY,
    high_speed_reward=HIGH_SPEED_REWARD,
)

# TODO: instanciar agente con hiperparams optimizados.

# El loop de entrenamiento debe respetar MAX_WALL_TIME_MIN: chequear
# (time.time() - t0) / 60 >= MAX_WALL_TIME_MIN al final de cada episodio
# (o cada N steps) y cortar. Si su agent.train() no acepta este cap como
# kwarg, puede envolverlo o pasarle un callback que levante una excepcion
# controlada cuando se exceda.
t0 = time.time()
# rewards = agent.train(..., max_wall_time_min=MAX_WALL_TIME_MIN)
elapsed_min = (time.time() - t0) / 60
train_env.close()

print(f"Tiempo de entrenamiento: {elapsed_min:.2f} min (cap = {MAX_WALL_TIME_MIN} min)")

In [ ]:
# Evaluacion deterministica: corremos N_EVAL_EPISODES greedy en un env con
# seed distinto al de training. El reward que va al ranking sale de aca, no
# del promedio movil de las ultimas ventanas de training (que tiene epsilon
# residual y depende del orden del buffer).

def evaluate(agent, n_episodes: int, eval_seed: int):
    eval_env = make_env(
        ENV_NAME,
        obs_height=OBS_HEIGHT,
        obs_width=OBS_WIDTH,
        stack_frames=NUM_STACKED_FRAMES,
        simulation_frequency=SIM_FREQUENCY_EVAL,
        seed=eval_seed,
        duration=EPISODE_DURATION,
        lanes_count=LANES_COUNT,
        vehicles_count=VEHICLES_COUNT,
        vehicles_density=VEHICLES_DENSITY,
        high_speed_reward=HIGH_SPEED_REWARD,
    )
    rewards = []
    for ep in range(n_episodes):
        # seed distinto por episodio para que los N tiren situaciones distintas
        state, _ = eval_env.reset(seed=eval_seed + ep)
        state = np.asarray(state, dtype=np.uint8)
        done = False
        ep_reward = 0.0
        while not done:
            # TODO: usar la politica greedy del agente (ej. agent.greedy_action(state)).
            action = agent.greedy_action(state)
            state, reward, terminated, truncated, _ = eval_env.step(action)
            state = np.asarray(state, dtype=np.uint8)
            ep_reward += float(reward)
            done = terminated or truncated
        rewards.append(ep_reward)
    eval_env.close()
    return rewards

# eval_rewards = evaluate(agent, N_EVAL_EPISODES, eval_seed=SEED + 9999)
# eval_mean = float(np.mean(eval_rewards))
# eval_std = float(np.std(eval_rewards))
# print(f"Eval greedy ({N_EVAL_EPISODES} eps): mean={eval_mean:.2f} ± {eval_std:.2f}")
# assert eval_mean >= TARGET_REWARD, f"{eval_mean:.2f} < {TARGET_REWARD}"


In [ ]:
# Reporte estructurado para que el ranking sea facil de extraer al re-ejecutar.
# Imprimir como JSON en una sola linea hace que se pueda grepear/parsear sin
# abrir el notebook.

# report = {
#     "technique": "...",          # ej. "DDQN+X+Y+Z"
#     "wall_time_min": round(elapsed_min, 2),
#     "wall_time_cap_min": MAX_WALL_TIME_MIN,
#     "eval_reward_mean": round(eval_mean, 2),
#     "eval_reward_std": round(eval_std, 2),
#     "n_eval_episodes": N_EVAL_EPISODES,
#     "target_reward": TARGET_REWARD,
#     "passed": eval_mean >= TARGET_REWARD,
#     "device": DEVICE,
#     "seed": SEED,
# }
# print(json.dumps(report, indent=2))

## Reporte

| Metrica | Valor |
| --- | --- |
| Tecnica | _TODO_ (DQN/DDQN o variante) |
| Reward final (eval mean ± std) | _TODO_ |
| Tiempo de entrenamiento | _TODO min_ |
| Hardware | _ORT / Colab T4_ |

## Mejoras aplicadas

Listar las mejoras introducidas respecto al baseline de la Parte 1, una por bullet. Para cada una:

- **Que se cambio** (ej. "replay buffer con Prioritized Experience Replay alpha=0.6, beta annealed 0.4 -> 1.0").
- **Por que** (que limitacion del baseline ataca: convergencia lenta, sobreestimacion de Q, exploracion pobre, throughput de env, etc.).
- **Impacto observado** (si midieron antes/despues: cuantos minutos ahorro, cuanto subio el reward, si estabilizo la curva). Si no llegaron a medir el delta de cada mejora aislada, decirlo explicitamente.

Mantener esta seccion breve (~media pagina). El detalle tecnico vive en el codigo y los comentarios; aca queremos justificar la receta, no re-explicar que es PER.

### Mejora 1 — _TODO_

_TODO: que / por que / impacto._

### Mejora 2 — _TODO_

_TODO_

### Mejora 3 — _TODO_ (opcional)

_TODO_
